# 04 · Failure-predictor ablation, and the chest-identity controls

Two jobs in one notebook because the second is meaningless without the first.

**Job 1 — reproduce the committed ladder.** `results/table-predictor-ablation.csv` is in the
repo; the code that produced it is not. That is a reproducibility hole in the result the
paper leans on hardest. §3 re-implements the ladder and **diffs against the committed
table**. If it cannot reproduce those numbers, the label definition or the CV protocol has
been reconstructed wrongly, and the notebook says so rather than appending numbers that
silently are not comparable.

**Job 2 — checklist item 1.** `biomedclip_full` (0.817) beats `biomedclip_crop` (0.750) at
t = 9.36. The crop contains the lesion. The full image contains the lesion *and the chest*.
A predictor that improves when handed more of the source radiograph may be identifying the
chest rather than the lesion. Source-grouped folds stop the same chest spanning folds; they
do not stop the *features* carrying chest identity. §4–§6 test that three ways.

---

## Read this before running

**The label definition is reconstructed, not recovered.** The original harness is gone. §2
scans candidate definitions and picks the one that reproduces the committed
`requested_axes` and `biomedclip_full` rows. Everything downstream inherits whatever §2
concludes, so read its output before trusting §3 onward.

**Checklist item 2 is partly ill-posed and this notebook does not pretend otherwise.**
"Does the detector score alone match 0.885?" — if failure is *defined* by thresholding
`det_edited`, then `det_edited` as a feature is circular and will score ~1.0. That is not a
finding. The non-circular versions of the same question are `det_background` (the detector's
score on the *unedited* chest, available before any edit) and `chest_onehot`, both of which
are in §3. `cnr` at 0.722 already answers the substance: a single hand-computed scalar is
statistically indistinguishable from the 512-dimensional lesion-crop embedding.

**CPU only.** No GPU, no generation. Roughly five minutes.

## 1 · Paths, data, and integrity

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install -q scikit-learn
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
NODE21    = ROOT/'01_data'/'00_source'/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

PRED_DIR.mkdir(parents=True, exist_ok=True)
for _p in (GRID_CSV, EMB_DIR/'synth_full.npz', EMB_DIR/'synth_crop.npz'):
    assert _p.exists(), f'{_p} missing -- see DRIVE_LAYOUT.md'
print('inputs ok')
print(f'writing to {PRED_DIR}')

In [ ]:
import numpy as np, pandas as pd

g = pd.read_csv(GRID_CSV, keep_default_na=False, na_values=[''])
print(f'grid: {len(g)} rows, {g.chest.nunique()} chests, {g.shape[1]} cols')
print(f"arms: {g.arm.value_counts().to_dict()}")
assert len(g) == 720 and g.shape[1] == 43, 'not the verified 720x43 grid'


def load_emb(path):
    """npz keyed by image_id. Handles the alternative (one matrix + an index) too, since
    both shapes have existed in this project."""
    z = np.load(path, allow_pickle=False)
    keys = list(z.keys())
    if len(keys) == 1 and z[keys[0]].ndim == 2:
        raise RuntimeError(f'{path.name} is a bare matrix with no ids -- needs an index csv')
    return {k: z[k].astype(np.float32).ravel() for k in keys}


E_full = load_emb(EMB_DIR/'synth_full.npz')
E_crop = load_emb(EMB_DIR/'synth_crop.npz')
print(f'\nsynth_full {len(E_full)} keys, dim {len(next(iter(E_full.values())))}')
print(f'synth_crop {len(E_crop)} keys, dim {len(next(iter(E_crop.values())))}')

ids = set(g.image_id)
for nm, E in [('synth_full', E_full), ('synth_crop', E_crop)]:
    miss = ids - set(E)
    assert not miss, f'{nm} missing {len(miss)} grid ids, e.g. {sorted(miss)[:3]}'
    extra = set(E) - ids
    if extra:
        print(f'  {nm}: {len(extra)} keys not in the grid (ignored)')

# duplicate-row check. An earlier feature file in this project held one vector repeated
# 180 times, and nothing computed from it could have been real.
for nm, E in [('synth_full', E_full), ('synth_crop', E_crop)]:
    M = np.stack([E[i] for i in g.image_id])
    u = len(np.unique(M, axis=0))
    print(f'  {nm}: {u}/{len(M)} unique rows' + ('' if u == len(M) else '   <-- DUPLICATES'))
    assert u == len(M), f'{nm} has duplicate embeddings'

## 2 · Recovering the label definition

The committed table is the only surviving record of what was scored. Two of its rows are
tight enough to identify the label:

- `requested_axes` is near chance, so its **AP ≈ the base rate**. Committed AP is
  **0.0716**, which implies roughly a 7% positive rate.
- `biomedclip_full` source-grouped AUROC is **0.8166**.

The scan below evaluates each candidate definition with the cheap `requested_axes` feature
and ranks by how close the base rate and AUROC land. A definition that misses on base rate
is the wrong definition no matter how good its AUROC looks.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

COMMITTED = pd.read_csv('/content/table-predictor-ablation.csv') \
    if Path('/content/table-predictor-ablation.csv').exists() else None
if COMMITTED is None:
    # not staged locally -- fall back to the two values we need, hard-coded from the repo
    COMMITTED = pd.DataFrame([
        dict(features='requested_axes',  mode='source_grouped', auroc_mean=0.5610342556,
             ap_mean=0.0716351141, brier_mean=0.2308188278),
        dict(features='biomedclip_full', mode='source_grouped', auroc_mean=0.8166447079,
             ap_mean=0.2616616328, brier_mean=0.1759931540),
        dict(features='biomedclip_crop', mode='source_grouped', auroc_mean=0.7500658762,
             ap_mean=0.1689538043, brier_mean=0.1070640248),
        dict(features='cnr',             mode='source_grouped', auroc_mean=0.7219367589,
             ap_mean=0.1879470744, brier_mean=0.2119111912),
        dict(features='edit_norm',       mode='source_grouped', auroc_mean=0.7265480896,
             ap_mean=0.1466351982, brier_mean=0.2170254976),
    ])
    print('using hard-coded committed values (upload table-predictor-ablation.csv to '
          '/content to diff against the full table)')
TARGET_AP  = float(COMMITTED.query("features=='requested_axes' and mode=='source_grouped'")
                   .ap_mean.iloc[0])
TARGET_AUC = float(COMMITTED.query("features=='biomedclip_full' and mode=='source_grouped'")
                   .auroc_mean.iloc[0])
print(f'targets -- base rate ~= {TARGET_AP:.4f}, biomedclip_full AUROC = {TARGET_AUC:.4f}')


def axes_features(df):
    return pd.get_dummies(df[['size', 'zone', 'overlap', 'side', 'position']],
                          drop_first=False).values.astype(float)


def quick_auc(X, y, groups, repeats=2):
    out = []
    for r in range(repeats):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=r)
        p = np.zeros(len(y), float)
        for tr, te in cv.split(X, y, groups):
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
            m.fit(X[tr], y[tr]); p[te] = m.predict_proba(X[te])[:, 1]
        out.append((roc_auc_score(y, p), average_precision_score(y, p)))
    return float(np.mean([a for a, _ in out])), float(np.mean([b for _, b in out]))


# each candidate: (name, subset mask, positive-class mask within that subset)
def build(name, sub, pos):
    return dict(name=name, sub=sub, pos=pos)

les = g.arm == 'lesion'
bg0 = g.det_background == 0
cands = [
    build('lesion, det==0',                    les,                     g.det_edited == 0),
    build('lesion & bg==0, det==0',            les & bg0,               g.det_edited == 0),
    build('lesion & bg==0, det<0.05',          les & bg0,               g.det_edited < 0.05),
    build('lesion & bg==0, det<0.5',           les & bg0,               g.det_edited < 0.5),
    build('lesion & bg==0 & en>0.5, det==0',   les & bg0 & (g.edit_norm > 0.5), g.det_edited == 0),
    build('lesion & bg==0 & en>1.0, det==0',   les & bg0 & (g.edit_norm > 1.0), g.det_edited == 0),
    build('lesion & bg==0 & cnr>0, det==0',    les & bg0 & (g.cnr > 0), g.det_edited == 0),
    build('lesion & bg==0 & gain>0, det<0.5',  les & bg0 & (g.gain > 0), g.det_edited < 0.5),
    build('all 720, det==0',                   pd.Series(True, index=g.index), g.det_edited == 0),
]

rows = []
for c in cands:
    sub = g[c['sub']].copy()
    y = c['pos'][c['sub']].values.astype(int)
    if y.sum() < 10 or y.sum() == len(y):
        rows.append(dict(definition=c['name'], n=len(sub), pos=int(y.sum()),
                         base_rate=np.nan, axes_auroc=np.nan, axes_ap=np.nan,
                         ap_err=np.nan)); continue
    auc, ap = quick_auc(axes_features(sub), y, sub.chest.values)
    rows.append(dict(definition=c['name'], n=len(sub), pos=int(y.sum()),
                     base_rate=round(y.mean(), 4), axes_auroc=round(auc, 4),
                     axes_ap=round(ap, 4), ap_err=round(abs(ap - TARGET_AP), 4)))

scan = pd.DataFrame(rows).sort_values('ap_err')
pd.set_option('display.width', 200)
print()
print(scan.to_string(index=False))
BEST = scan.dropna(subset=['ap_err']).iloc[0]
print(f"\nclosest on base rate: {BEST.definition}  (AP {BEST.axes_ap} vs target {TARGET_AP:.4f})")
print('\nIf no row lands within ~0.02 of the target AP, the label has NOT been recovered.')
print('Do not append new arms to the committed table in that case -- report the new arms')
print('as a standalone table with their own label definition stated.')

## 3 · The ladder, reproduced and diffed

Set `LABEL` from §2's verdict. `source_grouped` uses `StratifiedGroupKFold` on `chest`;
`stratified` is the leaky split, kept only to show how much it inflates.

The diff at the end is the point. A reproduction within ~0.02 AUROC means the protocol has
been recovered and new arms are comparable. Anything larger and they are not.

In [ ]:
# ---- set from the §2 scan -------------------------------------------------
# Selected by the §2 scan: base rate 0.0505 against a target of 0.0716, and the next
# candidate is off by 0.09. It also matches how the result is stated in docs/FINDINGS.md
# -- "detection among painted" -- so the original label was almost certainly "among
# lesions RadEdit actually painted, did the detector miss it".
# Re-read §2's output before trusting this; change it here if the scan disagrees.
LABEL_SUBSET = (g.arm == 'lesion') & (g.det_background == 0) & (g.edit_norm > 0.5)
LABEL_POS    = (g.det_edited == 0)          # positive class = detector missed the lesion
LABEL_NAME   = 'lesion & bg==0 & edit_norm>0.5, det==0'
# --------------------------------------------------------------------------

D = g[LABEL_SUBSET].reset_index(drop=True)
Y = LABEL_POS[LABEL_SUBSET].values.astype(int)
G = D.chest.values
print(f'label: {LABEL_NAME}')
print(f'n={len(D)}  positives={Y.sum()} ({Y.mean():.4f})  chests={D.chest.nunique()}')
print(f'positives per chest: {pd.Series(Y).groupby(G).sum().to_dict()}')

FEATURES = {
    'biomedclip_full':    lambda d: np.stack([E_full[i] for i in d.image_id]),
    'biomedclip_crop':    lambda d: np.stack([E_crop[i] for i in d.image_id]),
    'requested_axes':     axes_features,
    'cnr':                lambda d: d[['cnr']].values.astype(float),
    'edit_norm':          lambda d: d[['edit_norm']].values.astype(float),
    'edit_norm_plus_cnr': lambda d: d[['edit_norm', 'cnr']].values.astype(float),
    # --- new, non-circular arms ---
    'chest_onehot':       lambda d: pd.get_dummies(d.chest).values.astype(float),
    'det_background':     lambda d: d[['det_background']].values.astype(float),
    'structure_density':  lambda d: d[['structure_density', 'density_spread']].values.astype(float),
}

REPEATS, NSPLIT = 5, 5
OOF = {}          # (features, mode) -> pooled out-of-fold probabilities, last repeat


def evaluate(X, y, groups, mode, repeats=REPEATS):
    aur, aps, bri = [], [], []
    p = None
    for r in range(repeats):
        if mode == 'source_grouped':
            cv = StratifiedGroupKFold(n_splits=NSPLIT, shuffle=True, random_state=r)
            it = cv.split(X, y, groups)
        else:
            cv = StratifiedKFold(n_splits=NSPLIT, shuffle=True, random_state=r)
            it = cv.split(X, y)
        p = np.zeros(len(y), float)
        for tr, te in it:
            m = make_pipeline(StandardScaler(),
                              LogisticRegression(max_iter=5000, C=1.0))
            m.fit(X[tr], y[tr]); p[te] = m.predict_proba(X[te])[:, 1]
        aur.append(roc_auc_score(y, p)); aps.append(average_precision_score(y, p))
        bri.append(brier_score_loss(y, p))
    return dict(auroc_mean=np.mean(aur), auroc_repeat_sd=np.std(aur, ddof=1),
                ap_mean=np.mean(aps), ap_repeat_sd=np.std(aps, ddof=1),
                brier_mean=np.mean(bri), repeats=repeats), p


rows = []
for name, fn in FEATURES.items():
    X = fn(D)
    for mode in ('stratified', 'source_grouped'):
        stats, p = evaluate(X, Y, G, mode)
        OOF[(name, mode)] = p
        rows.append(dict(features=name, mode=mode, n_dim=X.shape[1], **stats))
        print(f'  {name:20} {mode:15} AUROC {stats["auroc_mean"]:.4f} '
              f'± {stats["auroc_repeat_sd"]:.4f}   AP {stats["ap_mean"]:.4f}   '
              f'Brier {stats["brier_mean"]:.4f}')

L = pd.DataFrame(rows)

In [ ]:
# ---- diff against the committed table ------------------------------------
chk = (L.merge(COMMITTED[['features', 'mode', 'auroc_mean', 'ap_mean']],
               on=['features', 'mode'], suffixes=('_new', '_committed')))
chk['auroc_delta'] = (chk.auroc_mean_new - chk.auroc_mean_committed).round(4)
chk['ap_delta']    = (chk.ap_mean_new - chk.ap_mean_committed).round(4)
print(chk[['features', 'mode', 'auroc_mean_new', 'auroc_mean_committed', 'auroc_delta',
           'ap_delta']].to_string(index=False))

worst = chk.auroc_delta.abs().max()
print(f'\nlargest AUROC deviation: {worst:.4f}')
if worst <= 0.02:
    print('REPRODUCED. The protocol is recovered and the new arms below are comparable.')
else:
    print('NOT REPRODUCED. The label or the CV protocol differs from the original run.')
    print('Report the new arms as a standalone table stating LABEL_NAME, and do not')
    print('present them alongside the committed numbers as though they were one ladder.')

## 4 · Control A — can chest identity alone predict failure?

The cheapest decisive test, and it needs no embedding. If a 12-dimensional one-hot of
`chest` predicts failure well, then failure rate varies substantially by chest, and *any*
model with access to chest information inherits that signal for free.

Note what `source_grouped` does to this arm: held-out chests are unseen, so their one-hot
columns are all zero at test time and the model can only fall back to the intercept. A high
`stratified` AUROC beside a `source_grouped` value near 0.5 is the signature of a feature
that works only by memorising chests — which is exactly the diagnosis being tested.

> The `source_grouped` number for `chest_onehot` is **degenerate, not a measurement.** Every
> test-fold prediction is the same constant, so the AUROC is decided by tie-breaking and can
> land either side of 0.5. Do not read a value below 0.5 as inverse signal, and do not
> report this cell as an arm of the ladder. The same caveat applies to `det_background`,
> which is zero for 693 of 720 rows and is therefore near-constant by construction.

In [ ]:
a = L.query("features=='chest_onehot'").set_index('mode')
print('chest_onehot:')
print(f"  stratified     AUROC {a.loc['stratified'].auroc_mean:.4f}   "
      '<- can memorise chests, they appear in both folds')
print(f"  source_grouped AUROC {a.loc['source_grouped'].auroc_mean:.4f}   "
      '<- held-out chests unseen, intercept only')

fail = pd.Series(Y, name='fail').groupby(G).agg(['sum', 'count'])
fail['rate'] = (fail['sum']/fail['count']).round(3)
print('\nfailure rate by chest:')
print(fail.sort_values('rate', ascending=False).to_string())
print(f"\nspread: {fail.rate.min():.3f} to {fail.rate.max():.3f}   "
      f"sd {fail.rate.std():.3f}")
print('\nA wide spread here is what a chest-aware predictor is able to exploit.')

## 5 · Control B — is chest identity decodable from the embedding?

Predict `chest` (12 classes) from `biomedclip_full` and from `biomedclip_crop`, with plain
stratified folds — grouping by chest is impossible when chest *is* the label.

High accuracy from the full image and low from the crop would mean the full-image embedding
carries chest identity that the crop does not. That, plus §4's spread in failure rate by
chest, is a complete mechanism for the 0.817 vs 0.750 gap without any lesion-difficulty
signal being involved.

In [ ]:
from sklearn.model_selection import cross_val_score

rows = []
for name in ('biomedclip_full', 'biomedclip_crop'):
    X = FEATURES[name](D)
    acc = cross_val_score(
        make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000)),
        X, G, cv=StratifiedKFold(5, shuffle=True, random_state=0),
        scoring='accuracy')
    rows.append(dict(features=name, chest_acc_mean=round(acc.mean(), 4),
                     chest_acc_sd=round(acc.std(ddof=1), 4)))
    print(f'  {name:18} chest-ID accuracy {acc.mean():.4f} ± {acc.std(ddof=1):.4f}')
CHEST_ID = pd.DataFrame(rows)
print(f'\nchance = 1/{D.chest.nunique()} = {1/D.chest.nunique():.4f}')
print('\nNear-perfect decoding means the embedding encodes which radiograph this is.')
print('That is not itself fatal -- but combined with §4 it means the failure signal')
print('could be entirely between-chest. §6 tests whether anything survives within a chest.')

## 6 · Control C — does anything survive *within* a chest?

The decisive test. Two views:

**Within-chest AUROC.** Take the source-grouped out-of-fold predictions and score them
separately inside each chest, then average. A predictor that ranks lesions by difficulty
works within a chest. A predictor that only ranks chests collapses to ~0.5 here.

**Chest-centred embeddings.** Subtract each chest's mean embedding, removing all
between-chest variation, and re-run. *Diagnostic only* — the centring uses every row's chest
mean including held-out ones, so it is not a reportable model. It answers one question: is
there within-chest signal at all?

In [ ]:
def within_chest_auc(p, y, groups):
    out = {}
    for c in np.unique(groups):
        m = groups == c
        if len(np.unique(y[m])) < 2:
            out[c] = np.nan; continue
        out[c] = roc_auc_score(y[m], p[m])
    s = pd.Series(out)
    return s

rows = []
for name in ('biomedclip_full', 'biomedclip_crop', 'cnr', 'edit_norm'):
    p = OOF[(name, 'source_grouped')]
    s = within_chest_auc(p, Y, G)
    rows.append(dict(features=name, pooled_auroc=round(roc_auc_score(Y, p), 4),
                     within_chest_mean=round(s.mean(), 4),
                     within_chest_sd=round(s.std(ddof=1), 4),
                     chests_scorable=int(s.notna().sum())))
    print(f'  {name:18} pooled {roc_auc_score(Y, p):.4f}   '
          f'within-chest {s.mean():.4f} ± {s.std(ddof=1):.4f}  '
          f'({s.notna().sum()} of {len(s)} chests have both classes)')
WITHIN = pd.DataFrame(rows)

print('\n--- chest-centred embeddings (diagnostic only) ---')
rows2 = []
for name in ('biomedclip_full', 'biomedclip_crop'):
    X = FEATURES[name](D)
    Xc = X - pd.DataFrame(X).groupby(G).transform('mean').values
    stats, _ = evaluate(Xc, Y, G, 'source_grouped')
    raw = float(L.query("features==@name and mode=='source_grouped'").auroc_mean.iloc[0])
    rows2.append(dict(features=name, auroc_raw=round(raw, 4),
                      auroc_chest_centred=round(stats['auroc_mean'], 4),
                      drop=round(raw - stats['auroc_mean'], 4)))
    print(f'  {name:18} raw {raw:.4f} -> chest-centred {stats["auroc_mean"]:.4f}   '
          f'(drop {raw-stats["auroc_mean"]:+.4f})')
CENTRED = pd.DataFrame(rows2)

print('\nREAD IT LIKE THIS:')
print('  within-chest ~0.5 AND a large centring drop  -> the 0.817 is between-chest')
print('     ranking. Checklist item 1 is confirmed and the predictor claim must be')
print('     restated as "which radiograph is hard", not "which lesion is hard".')
print('  within-chest clearly >0.5 AND a small drop   -> real within-chest difficulty')
print('     signal. Item 1 can be closed by reporting this table.')

## 7 · Save

In [ ]:
for df, nm in [(L, 'table-predictor-ablation-rerun.csv'),
               (chk, 'table-ablation-reproduction-diff.csv'),
               (scan, 'table-label-definition-scan.csv'),
               (CHEST_ID, 'table-chest-identity-decoding.csv'),
               (WITHIN, 'table-within-chest-auroc.csv'),
               (CENTRED, 'table-chest-centred-ablation.csv')]:
    df.to_csv(PRED_DIR/nm, index=False)
    print(f'  {nm:44} {len(df)} rows')

with open(PRED_DIR/'label_definition.txt', 'w') as f:
    f.write(f'LABEL_NAME: {LABEL_NAME}\n'
            f'n={len(D)} positives={int(Y.sum())} base_rate={Y.mean():.4f}\n'
            f'subset: arm==lesion & det_background==0\n'
            f'positive: det_edited == 0\n'
            f'reconstructed by the scan in section 2 -- NOT recovered from the '
            f'original harness\n')
print(f'\nall tables -> {PRED_DIR}')
print('\nNOT DONE HERE: k-NN coverage (checklist 9). It needs real failure labels, which')
print(f'now exist at {B1_DIR}/per_nodule.csv from the baselines run. Separate notebook.')